<a href="https://colab.research.google.com/github/ghroyd1110/Git-Commands/blob/Test/GeneratorExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 55.1 MB/s eta 0:00:00


In [2]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 20.3 MB/s eta 0:00:00


In [3]:
!pip install docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.4/451.4 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 55.4 MB/s eta 0:00:00
   ━━━━

In [6]:
# !pip install docling pandas
import os
import re
import json
import pandas as pd
from docling.document_converter import DocumentConverter
from google.colab import drive
from datetime import datetime

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# 2. CONFIGURATION
PDF_PATH = '/content/drive/MyDrive/PermitTest/3164_104799_ Permit_final.pdf'
JSON_OUTPUT = '/content/drive/MyDrive/PermitTest/Permit_NoSQL_Export.json'

def extract_to_nosql(pdf_path):
    print(f"Processing: {os.path.basename(pdf_path)}")
    converter = DocumentConverter()
    result = converter.convert(pdf_path)
    doc = result.document
    md_text = doc.export_to_markdown()

    # --- PART 1: GLOBAL METADATA ---
    def find_meta(label):
        pattern = f"{label}:?\\s*\\*?\\*?([^\\n|]+)"
        match = re.search(pattern, md_text, re.IGNORECASE)
        return match.group(1).strip().replace('*', '') if match else "N/A"

    permit_meta = {
        "permittee": find_meta("PERMITTEE"),
        "permit_no": find_meta("PERMIT No."),
        "place_id": find_meta("PLACE ID"),
        "date_issued": find_meta("DATE ISSUED"),
        "expiry_date": find_meta("EXPIRY DATE"),
        "source_file": os.path.basename(pdf_path)
    }

    # --- PART 2: DYNAMIC TABLE CAPTURE ---
    inventory = []

    # Iterate through every structural item identified by Docling
    for item, level in doc.iterate_items():
        if item.__class__.__name__ == "TableItem":
            # Convert table to a dictionary (NoSQL friendly)
            df = item.export_to_dataframe()

            # Clean up column names (remove newlines, extra spaces)
            df.columns = [str(c).strip().replace('\n', ' ') for c in df.columns]

            # Convert each row to a dictionary
            rows = df.to_dict(orient='records')

            for row in rows:
                # Create a master document for this piece of equipment
                equipment_doc = {
                    "metadata": permit_meta,
                    "specs": {k: v for k, v in row.items() if pd.notna(v)} # Only keep non-empty fields
                }

                # Logic-based tagging (Inferred Fields)
                row_str = str(row).upper()
                equipment_doc["inferred"] = {
                    "fuel_type": "Diesel" if "DIESEL" in row_str else ("Natural Gas" if "GAS" in row_str else "TBD"),
                    "status": "Emergency" if "EMERGENCY" in row_str else "Standard",
                    "category": "Generator" if "GEN" in row_str else ("Boiler" if "BOIL" in row_str else "Other")
                }

                inventory.append(equipment_doc)

    return inventory

# --- EXECUTION ---
full_inventory = extract_to_nosql(PDF_PATH)

# Save as a JSON line file (best for NoSQL databases like MongoDB)
# Ensure you are using json.dumps to create the line
# When saving your entries to the JSONL file, use these exact settings:
# Use this to write your JSONL file
with open('Permit_NoSQL_Export.json', 'w', encoding='utf-8') as f:
    for entry in full_inventory:
        # json.dumps handles the "inch marks" escaping automatically
        f.write(json.dumps(entry) + '\n')

print(f"Successfully exported {len(full_inventory)} equipment documents to JSON.")

[INFO] 2026-04-01 23:21:34,058 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-01 23:21:34,059 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-01 23:21:34,108 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-04-01 23:21:34,109 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing: 3164_104799_ Permit_final.pdf


[INFO] 2026-04-01 23:21:34,345 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-01 23:21:34,347 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-01 23:21:34,353 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-04-01 23:21:34,355 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-04-01 23:21:34,451 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-01 23:21:34,452 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-01 23:21:34,553 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.pth
[INFO] 2026-04-01 23:21:34,554 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.pth


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Successfully exported 171 equipment documents to JSON.
